In [ ]:
from IPython.core.display import HTML
HTML("""
    <style>
    body { font-feature-settings: "liga" 0; }
    </style>
""")

# Hot Shower
The objective of this demo is to illustrate how a parametric model can be constructed without data, and then subsequently linked to data sources in the cloud.

In the process, we will illustrate several features of LCA modeling in Antelope:
 - model construction with *observable* parameters
 - nesting and anchoring
 - linking
 - scenario-based LCIA


In [ ]:
from antelope_foreground import ForegroundCatalog
from antelope import enum


In [ ]:
from antelope_reports import QuickAndEasy

In [ ]:
cat = ForegroundCatalog()

In [ ]:
fg = cat.create_foreground('demo')

In [ ]:
Q = QuickAndEasy(fg)

#

In [ ]:
help(Q.new_link)

## Step 1- the shower
First thing you do is take a shower, right? How do we want to parameterize this activity?
 - Duration of shower
 - Water flow rate
 - temperature of water
 - efficiency of heater
 - ... soap?


In [ ]:
volume = cat.get_canonical('volume')
mass = cat.get_canonical('mass')
ncv = cat.get_canonical('net calorific value')

In [ ]:
volume.get('UnitConversion')

In [ ]:
shower = Q.new_link('hot shower', 'Count', 'Output')

In [ ]:
shower.show_tree()

In [ ]:
water_min = Q.new_link('duration of shower (minutes)', 'Count', 'Input', parent=shower, amount=15)
soap = Q.new_link('soap', 'volume', 'Input', parent=shower, amount=3, units='mL')

In [ ]:
water_vol = Q.new_link('Flow rate per minute', 'volume', 'Input', parent=water_min, amount=2, units='gal')

In [ ]:
water_supply = Q.new_link('water supply', 'volume', 'Input', balance=True, parent=water_vol)
temp_increase = Q.new_link('temperature increase degrees C', 'Count', 'Input', parent=water_vol, amount=40)
heat_input = Q.new_link('heat input required', 'net calorific value', 'Input', parent=temp_increase, amount=4.184, units='MJ')

In [ ]:
shower.show_tree()

In [ ]:
shower.show_tree(observed=True)

In [ ]:
_=enum(shower.cutoffs(True))

In [ ]:
from antelope_reports.charts.model_graph import ModelGraph

In [ ]:
g = ModelGraph(shower)

In [ ]:
g.display()

In [ ]:
from IPython.display import Image, display

# add if using jupyter
%matplotlib inline  

def view_pydot(pdot):
    plt = Image(pdot.graph.create_png())
    display(plt)

In [ ]:
view_pydot(g)

## Heat provisioning
Right now we have a model with three cut-off flows: soap, water, and heat.  Of those, the heat is the most complex and likely the most impactful.  Let's make a couple of different heat provisioning models, one with natural gas and the other with electricity.


In [ ]:
heat_ng = Q.new_link('Heat from natural gas', ncv, 'Output')
heat_ng.observe(0.65)
heat_in = Q.new_link('Heat combustion', ncv, 'Input', parent=heat_ng, amount=1.0)

In [ ]:
heat_ng.show_tree(True)

In [ ]:
heat_elec = Q.new_link('Heat from electricity', ncv, 'Output')
heat_elec.observe(0.85)
heat_in_e = Q.new_link('Heat in from electricity', ncv, 'Input', parent=heat_elec, amount=1.0)


In [ ]:
fg.observe(heat_input, anchor_node=heat_ng, scenario='natural gas')
fg.observe(heat_input, anchor_node=heat_elec, scenario='electric')

In [ ]:
_=enum(shower.cutoffs('electric'))

In [ ]:
g = ModelGraph(shower, scenario='electric')
view_pydot(g)

## Linking the model

So now we have a data-free model that describes the foreground activity of taking a shower.  We need to fill this model out with details about the supply of the products and resources we require.  In Antelope terms, we need to anchor our cutoff flows.

To do that, we'll use data sets from the Federal LCA Commons.

In [ ]:
cat.blackbook_guest('https://sc.vault.lc')

In [ ]:
list(cat.blackbook_origins)

In [ ]:
cat.get_blackbook_resources('lcacommons.uslci.fy24.q1.01')[0]

In [ ]:
q_us = cat.query('lcacommons.uslci')

In [ ]:
ngs = enum(q_us.flows(name='natural gas'))

In [ ]:
ng = ngs[16]

In [ ]:
tgts = enum(ng.targets())

In [ ]:
n = tgts[0]
str(n.reference())

In [ ]:
heat_ng.show_tree()

In [ ]:
help(heat_in.flow.characterize)

In [ ]:
heat_in.flow.characterize('volume', 1.357/52)

In [ ]:
fg.observe(heat_in, anchor_node=n)

In [ ]:
heat_ng.show_tree(True)

In [ ]:
ffs=enum(heat_ng.traverse(True))

In [ ]:
ffs[1]

In [ ]:
_=enum(shower.cutoffs('natural gas'))

In [ ]:
elecs = enum(q_us.flows(name='electricity'))

In [ ]:
elecs = enum(q_us.processes(name='electricity, at grid'))

In [ ]:
elec = elecs[20]
elec.show()
str(elec.reference())

In [ ]:
heat_in_e.show()

In [ ]:
fg.observe(heat_in_e, anchor_node=elec)

In [ ]:
heat_elec.show_tree()

## Let's run some numbers
We still need an LCIA method-- we'll grab it from blackbook*

(* soon to be replaced with Qdb!)

In [ ]:
cat.get_blackbook_resources('lcia.traci.2.1')
q_t = cat.query('lcia.traci')

In [ ]:
qs = enum(q_t.lcia_methods())

In [ ]:
gwp = qs[4]

In [ ]:
shower.fragment_lcia(gwp).show_components()

In [ ]:
shower.fragment_lcia(gwp, scenario='natural gas').show_components()

In [ ]:
shower.fragment_lcia(gwp, scenario='electric').show_components()

In [ ]:
fg.observe(heat_in_e, anchor_node=elecs[13], scenario='WECC')

In [ ]:
shower.fragment_lcia(gwp, scenario=('electric', 'WECC')).show_components()

### rollup
We can change our foreground model to conceal the source of electricity if we want, by specifying the 'descend' flag on the anchor.


In [ ]:
heat_input.termination('electric').descend = False

In [ ]:
shower.fragment_lcia(gwp, scenario=('electric', 'WECC')).show_components()